In [3]:
class Node:
    def __init__(self, state, parent=None, g=0, h=0):
        self.state = state
        self.parent = parent
        self.g = g  # Cost to reach this node (depth)
        self.h = h  # Heuristic cost to reach the goal
        self.f = g + h  # Total cost

# Function to calculate the number of misplaced tiles
def misplaced_tiles(state, goal_state):
    return sum(1 for i in range(9) if state[i] != goal_state[i] and state[i] != 0)

# Function to calculate the Manhattan distance
def manhattan_distance(state, goal_state):
    distance = 0
    for i in range(9):
        if state[i] != 0:
            current_pos = i
            goal_pos = goal_state.index(state[i])
            distance += abs(current_pos % 3 - goal_pos % 3) + abs(current_pos // 3 - goal_pos // 3)
    return distance

# Function to get the possible neighbors by moving the blank space
def get_neighbors(state):
    neighbors = []
    blank_index = state.index(0)
    x, y = blank_index % 3, blank_index // 3

    directions = {
        "up": (x, y - 1),
        "down": (x, y + 1),
        "left": (x - 1, y),
        "right": (x + 1, y),
    }

    for direction, (new_x, new_y) in directions.items():
        if 0 <= new_x < 3 and 0 <= new_y < 3:
            new_blank_index = new_y * 3 + new_x
            new_state = state[:]
            new_state[blank_index], new_state[new_blank_index] = new_state[new_blank_index], new_state[blank_index]
            neighbors.append(new_state)

    return neighbors

# A* search algorithm using custom data structures
def a_star(maze, start_state, goal_state, heuristic):
    open_list = []  # List to store nodes to be explored
    closed_list = []  # List to store explored nodes
    open_list.append(Node(start_state, None, 0, heuristic(start_state, goal_state)))

    while open_list:
        # Find the node with the smallest f in open_list
        current_node = min(open_list, key=lambda node: node.f)
        open_list.remove(current_node)
        closed_list.append(current_node)

        if current_node.state == goal_state:
            path = []
            steps = 0
            while current_node is not None:
                path.append(current_node.state)
                current_node = current_node.parent
                steps += 1
            return path[::-1], steps - 1  # Reversed path and step count

        # Generate children (neighboring states)
        children = []
        neighbors = get_neighbors(current_node.state)

        for neighbor in neighbors:
            children.append(Node(neighbor, current_node))

        for child in children:
            if any(closed_child.state == child.state for closed_child in closed_list):
                continue

            child.g = current_node.g + 1  # g(n) + distance between current and child (1 in this case)
            child.h = heuristic(child.state, goal_state)
            child.f = child.g + child.h

            if any(open_node.state == child.state and child.g >= open_node.g for open_node in open_list):
                continue

            open_list.append(child)

    return None, None

# Example Usage
maze = None  # Placeholder, not needed for the 8-puzzle
initial_state = [2, 8, 3, 1, 6, 4, 7, 0, 5]  # Example start state
goal_state = [1, 2, 3, 8, 0, 4, 7, 6, 5]     # Goal state

# Using Misplaced Tiles heuristic
path_1, steps_1 = a_star(maze, initial_state, goal_state, misplaced_tiles)
print("Path using Misplaced Tiles Heuristic:")
for state in path_1:
    print(state[:3])
    print(state[3:6])
    print(state[6:])
    print()
print("Step Count:", steps_1)

# Using Manhattan Distance heuristic
path_2, steps_2 = a_star(maze, initial_state, goal_state, manhattan_distance)
print("Path using Manhattan Distance Heuristic:")
for state in path_2:
    print(state[:3])
    print(state[3:6])
    print(state[6:])
    print()
print("Step Count:", steps_2)


Path using Misplaced Tiles Heuristic:
[2, 8, 3]
[1, 6, 4]
[7, 0, 5]

[2, 8, 3]
[1, 0, 4]
[7, 6, 5]

[2, 0, 3]
[1, 8, 4]
[7, 6, 5]

[0, 2, 3]
[1, 8, 4]
[7, 6, 5]

[1, 2, 3]
[0, 8, 4]
[7, 6, 5]

[1, 2, 3]
[8, 0, 4]
[7, 6, 5]

Step Count: 5
Path using Manhattan Distance Heuristic:
[2, 8, 3]
[1, 6, 4]
[7, 0, 5]

[2, 8, 3]
[1, 0, 4]
[7, 6, 5]

[2, 0, 3]
[1, 8, 4]
[7, 6, 5]

[0, 2, 3]
[1, 8, 4]
[7, 6, 5]

[1, 2, 3]
[0, 8, 4]
[7, 6, 5]

[1, 2, 3]
[8, 0, 4]
[7, 6, 5]

Step Count: 5


In [3]:
from collections import deque

def bfs(matrix, start, goal):
    n = len(matrix)
    directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]  # up, down, left, right
    visited = set()
    queue = deque([(start, [])])  # (current_position, path_taken)
    visited.add(start)
    
    while queue:
        (x, y), path = queue.popleft()
        
        if (x, y) == goal:
            return path
        
        for dx, dy in directions:
            nx, ny = x + dx, y + dy
            if 0 <= nx < n and 0 <= ny < n and (nx, ny) not in visited:
                visited.add((nx, ny))
                queue.append(((nx, ny), path + [(nx, ny)]))
    
    return []

def collect_gold(matrix, start, gold_positions):
    n = len(matrix)
    total_path = []
    current_position = start

    for gold in gold_positions:
        # Step 1: Move to gold position
        path_to_gold = bfs(matrix, current_position, gold)
        total_path.extend(path_to_gold)
        current_position = gold
        
        # Step 2: Move to (0, 0) after collecting the gold
        path_to_origin = bfs(matrix, current_position, (0, 0))
        total_path.extend(path_to_origin)
        current_position = (0, 0)
    
    return total_path

# Example usage
matrix = [
    [0, 1, 0, 0],
    [0, 0, 0, 1],
    [1, 0, 1, 0],
    [0, 0, 0, 0],
]

start = (3, 3)  # starting point
gold_positions = [(0, 1), (1, 3), (2, 0), (2, 2)]  # order of gold collection

steps = collect_gold(matrix, start, gold_positions)

# Printing steps
for step in steps:
    print(f"Move to {step}")


Move to (2, 3)
Move to (1, 3)
Move to (0, 3)
Move to (0, 2)
Move to (0, 1)
Move to (0, 0)
Move to (1, 0)
Move to (1, 1)
Move to (1, 2)
Move to (1, 3)
Move to (0, 3)
Move to (0, 2)
Move to (0, 1)
Move to (0, 0)
Move to (1, 0)
Move to (2, 0)
Move to (1, 0)
Move to (0, 0)
Move to (1, 0)
Move to (2, 0)
Move to (2, 1)
Move to (2, 2)
Move to (1, 2)
Move to (0, 2)
Move to (0, 1)
Move to (0, 0)


In [6]:
class Node:
    def __init__(self, state, parent=None, g=0, h=0):
        self.state = state
        self.parent = parent
        self.g = g  #  (depth)
        self.h = h  # Heuristic cost 
        self.f = g + h  # Total cost


def misplaced_tiles(state, goal_state):
    return sum(1 for i in range(9) if state[i] != goal_state[i] and state[i] != 0)

def manhattan_distance(state, goal_state):
    distance = 0
    for i in range(9):
        if state[i] != 0:
            current_pos = i
            goal_pos = goal_state.index(state[i])
            distance += abs(current_pos % 3 - goal_pos % 3) + abs(current_pos // 3 - goal_pos // 3)
    return distance

def get_neighbors(state):
    neighbors = []
    blank_index = state.index(0)
    x, y = blank_index % 3, blank_index // 3

    directions = {
        "up": (x, y - 1),
        "down": (x, y + 1),
        "left": (x - 1, y),
        "right": (x + 1, y),
    }

    for direction, (new_x, new_y) in directions.items():
        if 0 <= new_x < 3 and 0 <= new_y < 3:
            new_blank_index = new_y * 3 + new_x
            new_state = state[:]
            new_state[blank_index], new_state[new_blank_index] = new_state[new_blank_index], new_state[blank_index]
            neighbors.append(new_state)

    return neighbors

def a_star(maze, start_state, goal_state, heuristic):
    open_list = [] 
    closed_list = []  
    open_list.append(Node(start_state, None, 0, heuristic(start_state, goal_state)))

    while open_list:
        current_node = min(open_list, key=lambda node: node.f)
        open_list.remove(current_node)
        closed_list.append(current_node)

        if current_node.state == goal_state:
            path = []
            steps = 0
            while current_node is not None:
                path.append(current_node.state)
                current_node = current_node.parent
                steps += 1
            return path[::-1], steps - 1  

        children = []
        neighbors = get_neighbors(current_node.state)

        for neighbor in neighbors:
            children.append(Node(neighbor, current_node))

        for child in children:
            if any(closed_child.state == child.state for closed_child in closed_list):
                continue

            child.g = current_node.g + 1  
            child.h = heuristic(child.state, goal_state)
            child.f = child.g + child.h

            if any(open_node.state == child.state and child.g >= open_node.g for open_node in open_list):
                continue

            open_list.append(child)

    return None, None

def get_user_input_matrix(prompt):
    print(prompt)
    state = []
    for i in range(3):
        while True:
            try:
                row = input(f"Enter row {i + 1} (space-separated numbers, 0 for blank): ").split()
                if len(row) == 3 and all(int(num) in range(9) for num in row) and len(set(row)) == 3:
                    state.extend([int(num) for num in row])
                    break
                else:
                    print("Invalid input. Please enter three unique numbers between 0 and 8.")
            except ValueError:
                print("Invalid input. Please enter numbers.")
    return state

# Main function
def main():
    initial_state = get_user_input_matrix("Enter the initial state of the puzzle (row by row):")
    goal_state = get_user_input_matrix("Enter the goal state of the puzzle (row by row):")

    print("\nChoose the heuristic function:")
    print("1. Number of Misplaced Tiles")
    print("2. Manhattan Distance")
    heuristic_choice = input("Enter 1 or 2: ")

    if heuristic_choice == "1":
        heuristic = misplaced_tiles
    elif heuristic_choice == "2":
        heuristic = manhattan_distance
    else:
        print("Invalid choice. Defaulting to Manhattan Distance.")
        heuristic = manhattan_distance

    maze = None 
    path, steps = a_star(maze, initial_state, goal_state, heuristic)


    if path:
        print("\nSolution Path:")
        for state in path:
            print(state[:3])
            print(state[3:6])
            print(state[6:])
            print()
        print(f"Total Steps: {steps}")
    else:
        print("No solution found.")

if __name__ == "__main__":
    main()


Welcome to the 8-Puzzle Solver using A* Algorithm!
Enter the initial state of the puzzle (row by row):
Enter row 1 (space-separated numbers, 0 for blank): 2 8 3
Enter row 2 (space-separated numbers, 0 for blank): 1 6 4
Enter row 3 (space-separated numbers, 0 for blank): 7 0 5
Enter the goal state of the puzzle (row by row):
Enter row 1 (space-separated numbers, 0 for blank): 1 2 3
Enter row 2 (space-separated numbers, 0 for blank): 8 0 4
Enter row 3 (space-separated numbers, 0 for blank): 7 6 5

Choose the heuristic function:
1. Number of Misplaced Tiles
2. Manhattan Distance
Enter 1 or 2: 2

Solution Path:
[2, 8, 3]
[1, 6, 4]
[7, 0, 5]

[2, 8, 3]
[1, 0, 4]
[7, 6, 5]

[2, 0, 3]
[1, 8, 4]
[7, 6, 5]

[0, 2, 3]
[1, 8, 4]
[7, 6, 5]

[1, 2, 3]
[0, 8, 4]
[7, 6, 5]

[1, 2, 3]
[8, 0, 4]
[7, 6, 5]

Total Steps: 5


In [12]:
class PriorityQueue:
    def __init__(self):
        self.elements = []
    
    def is_empty(self):
        return len(self.elements) == 0
    
    def put(self, item, priority):
        self.elements.append((priority, item))
    
    def get(self):
        min_index = 0
        for i in range(len(self.elements)):
            if self.elements[i][0] < self.elements[min_index][0]:
                min_index = i
        return self.elements.pop(min_index)[1]

def astar(grid, start, goal):

    directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]

    def manhattan_dist(point, goal):
        return abs(point[0] - goal[0]) + abs(point[1] - goal[1])

    open_list = PriorityQueue()
    open_list.put((start, 0), manhattan_dist(start, goal))

    came_from = {}
    g_score = {start: 0}
    
    while not open_list.is_empty():
        current = open_list.get()
        current_pos = current[0]
        
        if current_pos == goal:

            path = []
            while current_pos:
                path.append(current_pos)
                current_pos = came_from.get(current_pos)
            return path[::-1]
        
        for d in directions:
            neighbor = (current_pos[0] + d[0], current_pos[1] + d[1])
            if (0 <= neighbor[0] < len(grid)) and (0 <= neighbor[1] < len(grid[0])):
                if grid[neighbor[0]][neighbor[1]] != 1:  # 1 represents obstacles
                    tentative_g_score = g_score[current_pos] + 1
                    if neighbor not in g_score or tentative_g_score < g_score[neighbor]:
                        g_score[neighbor] = tentative_g_score
                        f_score = tentative_g_score + manhattan_dist(neighbor, goal)
                        open_list.put((neighbor, f_score), f_score)
                        came_from[neighbor] = current_pos
    
    return None  # No path found


rows = int(input("Enter number of rows: "))
cols = int(input("Enter number of columns: "))


grid = []
print("Enter the grid (0 for free space, 1 for obstacle):")
for i in range(rows):
    grid.append(list(map(int, input().split())))


start = tuple(map(int, input("Enter start position (row column): ").split()))
goal = tuple(map(int, input("Enter goal position (row column): ").split()))

# Execute A* algorithm
path = astar(grid, start, goal)


if path:
    print("Path found:", path)
else:
    print("No path found.")


Enter number of rows: 9
Enter number of columns: 9
Enter the grid (0 for free space, 1 for obstacle):
0 0 0 0 0 0 0 0 0
0 0 0 0 1 0 0 0 0
0 0 0 1 1 1 0 0 0
0 0 0 0 0 1 0 0 0 
0 1 1 0 1 1 1 0 0
0 1 0 0 1 1 0 0 0
0 1 1 0 0 1 1 0 0
0 0 0 0 0 0 0 1 0
0 0 0 0 0 0 0 0 0
Enter start position (row column): 0 0 
Enter goal position (row column): 4 8
Path found: [(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (0, 5), (1, 5), (1, 6), (2, 6), (3, 6), (3, 7), (4, 7), (4, 8)]


In [ ]:
0 0 0 0 0 0 0 0 0
0 0 0 0 1 0 0 0 0 
0 0 0 1 1 1 0 0 0 
0 0 0 0 0 1 0 0 0 
0 1 1 0 1 1 1 0 0
0 1 0 0 1 1 0 0 0 
0 1 1 0 0 1 1 0 0 
0 0 0 0 0 0 0 1 0
0 0 0 0 0 0 0 0 0